# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide to loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library. All dataset components (record sets, fields, columns) are accessed by their persistent `@id`s, ensuring clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print the dataset's name and description as attributes
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs for this dataset.

Below, we enumerate all record sets and for each their fields and columns, referencing all entities by their `@id`.

In [ ]:
# List all record sets by @id and inspect their fields/columns
record_sets = list(dataset.record_sets.values())

if len(record_sets) == 0:
    print("No record sets defined in this dataset.")
else:
    for rset in record_sets:
        print(f"RecordSet: {rset.id}")
        print(f"  Name: {getattr(rset, 'name', '[No name]')}")
        print("  Fields (by @id):")
        for field in rset.fields:
            print(f"    - {field.id}")
            print(f"      Name: {getattr(field, 'name', '[No name]')}")
            print(f"      DataType: {getattr(field, 'data_type', '[No data_type]')}")
            if hasattr(field, 'columns'):
                print(f"      Columns (@id): {[col.id for col in field.columns]}")
        print("\n-----------------------------\n")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

All extractions below use record set and field `@id` values from above.

If the dataset only contains a single record set, it will be loaded below. If there are multiple, all are loaded by their `@id`.

In [ ]:
dataframes = {}

# Obtain all record set IDs
record_set_ids = list(dataset.record_sets.keys())
print(f"Record set @ids found: {record_set_ids}")

if len(record_set_ids) == 0:
    print("No record sets to extract data from.")
else:
    # Load each record set as a DataFrame, using its @id
    for rs_id in record_set_ids:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set: {rs_id} with {len(dataframes[rs_id])} records.")

    # Show columns for the first available record set
    first_rs_id = record_set_ids[0]
    print(f"Available columns (@id) in '{first_rs_id}':\n{list(dataframes[first_rs_id].columns)}\n")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll select one numeric field (by its `@id`) for filtering and normalization. We also demonstrate a groupby operation on a preferred categorical field if available.

You may wish to adjust `numeric_field_id` and `group_field_id` according to your dataset's contents.

In [ ]:
# Specify record set and relevant field @ids
record_set_id = record_set_ids[0]  # Use the first available record set
df = dataframes[record_set_id]

# List available columns for easy selection
print(f"Columns available in DataFrame:")
for col in df.columns:
    print(f" - {col}")

# Select a likely numeric field by @id (edit this as appropriate)
numeric_field_id = None
candidate_numeric = [c for c in df.columns if ('age' in c.lower() or 'interval' in c.lower() or 'met' in c.lower() or 'num' in c.lower())]
if candidate_numeric:
    numeric_field_id = candidate_numeric[0]
else:
    # Fallback: pick the first column
    numeric_field_id = df.columns[0]
print(f"Using numeric field (@id): {numeric_field_id}")

# Filter records with value above a threshold (example: >50)
threshold = 50
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print(f"Field '{numeric_field_id}' is not numeric. Please select a numeric field for EDA.")

# Attempt to group by a categorical field (example: one containing 'sex', 'location', 'site', or 'status')
group_field_id = None
group_candidates = [c for c in df.columns if any(x in c.lower() for x in ['sex', 'location', 'site', 'status', 'type']) and c != numeric_field_id]
if group_candidates:
    group_field_id = group_candidates[0]
    print(f"Grouping by field: {group_field_id}")
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped means by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize a numeric data distribution and boxplot by group (if suitable fields present).

_Note: Adjust the field @ids below if desired._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
if numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
else:
    print(f"Field '{numeric_field_id}' not in DataFrame columns.")

# If both numeric and group fields exist, show boxplot
if group_field_id and numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
This notebook guided you through:
- Loading a structured Croissant dataset and reading its metadata
- Listing all record sets, fields, and columns by their `@id`
- Extracting tabular data for record sets and referencing all fields by persistent IDs
- Conducting basic EDA and visualizations with normalization, groupby, and plotting

To extend this notebook:
- Swap field `@id`s for others of interest in your dataset.
- Add further custom analyses, interactive widgets, or export results to files for reporting.